In [ ]:
# ===========================================================================
# RESULTS - the project's FINAL REPORTING GATE
#
# This notebook decides what may be claimed. Nothing here recomputes a
# metric; it aggregates, validates, tests, and exports what
# src/training/reporting.py already wrote per run.
#
#   src/results.py   experiment registry reader, eligibility gate,
#                     paired significance testing (Wilcoxon, McNemar,
#                     bootstrap CI), ablation gains, ROC/PR overlay,
#                     gated export
#
# WHY THIS NOTEBOOK HAS A GATE
# It previously concatenated every CSV under outputs/metrics/ and formatted
# whatever it found. That is how a single held-out healthy-control speaker -
# one fold of an intended 28, with no positive class at all - became a
# headline "99.7% accuracy" row, and how seven one-fold benchmark runs
# appeared beside real experiments. A file existing is not evidence that an
# experiment happened.
#
# Every run is now sorted into exactly one tier, with a stated reason:
#
#   FINAL        complete coverage, both classes represented in the pooled
#                held-out set. Claims may rest on these.
#   PRELIMINARY  real but incomplete. Shown, labelled, never claimed.
#   EXCLUDED     cannot support a claim - never ran, single-class held-out
#                set, or a diagnostic/benchmark run.
#
# Nothing is deleted and nothing is hidden. Incomplete work stays visible;
# it just cannot masquerade as a result.
#
# PREREQUISITES: at least one REGISTERED run from notebooks/03_training.ipynb.
# Runs from before the registry existed will not appear - re-run to register.
# ===========================================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv, print_note, print_table
from src.results import build_result_tiers, set_publication_style
from src.training.reporting import summarize_registry

config.ensure_directories()
set_publication_style()

# A FINAL result must have reached every fold of its intended protocol.
# Lower this only deliberately, and say so in the write-up.
MIN_COVERAGE = 1.0

registry_summary = summarize_registry()
tiers = build_result_tiers(MIN_COVERAGE)

print_header("Results — final reporting gate")
print_kv("Metrics directory", config.METRICS_DIR)
print_kv("Results directory", config.RESULTS_DIR)
print_kv("Minimum coverage for FINAL", f"{MIN_COVERAGE:.0%}")

if registry_summary.empty:
    print_note("The experiment registry is EMPTY. No run has been executed since "
              "the registry was added, so there is nothing to report. Pre-repair "
              "runs are unregistered by design - their validity could not be "
              "established retrospectively, which is the problem the registry "
              "exists to solve. Re-run notebooks/03_training.ipynb to populate it.")
else:
    print_kv("Registered runs", len(registry_summary))
    print_kv("FINAL / PRELIMINARY / EXCLUDED",
             f"{len(tiers['final'])} / {len(tiers['preliminary'])} / {len(tiers['excluded'])}")

In [ ]:
# STAGE 1 - The three result tables.
#
# Read them in order. The FINAL table is the only one a claim may rest on;
# the other two exist so that incomplete and invalid work stays visible
# instead of quietly vanishing from the report.
#
# Undefined metrics appear as NaN (exported as "N/A"), never as 0 - see
# src/training/metrics.py. On a single-class held-out fold there is no
# positive class to have found or missed, so precision/recall/F1/AUROC have
# no value at all; writing 0 there was the pipeline's most misleading bug.
from src.results import (build_final_results_table, build_preliminary_results_table,
                         style_comparison_table)

final_results = build_final_results_table(MIN_COVERAGE)
preliminary_results = build_preliminary_results_table(MIN_COVERAGE)
excluded_results = tiers["excluded"]

print_header("FINAL RESULTS — eligible for reporting")
if final_results.empty:
    print_note("No run has passed the eligibility gate. A run reaches FINAL only "
              "with complete fold coverage AND both classes present in the pooled "
              "held-out set. Until then this project has no reportable result, "
              "which is an honest statement of where the experiments stand.")
else:
    score_cols = ["accuracy", "precision", "recall", "specificity", "f1", "auroc"]
    display(final_results[["run_name", "task", "completed_folds", "expected_folds",
                           "coverage"] + score_cols].style.format(
        {c: "{:.4f}" for c in score_cols + ["coverage"]}, na_rep="N/A"))

print_header("PRELIMINARY RESULTS — real but incomplete, NOT FOR FINAL CLAIMS")
if preliminary_results.empty:
    print_kv("Preliminary runs", "none")
else:
    print_table(preliminary_results[["run_name", "task", "completed_folds",
                                     "expected_folds", "accuracy", "f1", "auroc",
                                     "reason"]])
    print_note("These runs did not reach their intended fold count. Their numbers "
              "describe only the folds that ran and are not comparable against a "
              "complete run, or against each other if their coverage differs.")

print_header("EXCLUDED — cannot support a claim")
if excluded_results.empty:
    print_kv("Excluded runs", "none")
else:
    print_table(excluded_results[["run_name", "task", "completed_folds",
                                  "expected_folds", "reason"]])

In [ ]:
# STAGE 2 - Statistical significance tests. Is a fusion refinement actually
# better than the model it claims to improve on, or within LOSO fold noise?
# Paired Wilcoxon signed-rank test, fold-matched, on F1 - the same metric
# notebooks/03_training.ipynb's severity gating ranks by. Compares the two
# Phase 6 attention-fusion variants (Models E, F) against the plain
# concatenated 'fusion' run (Model D), since that is the pre-attention
# baseline each of them claims to improve on.
from src.results import compare_models_statistically

BASELINE_RUN = "detection_fusion"
CANDIDATE_RUNS = ["detection_attention_fusion", "detection_attention_fusion_praat"]

significance_results = []
for candidate in CANDIDATE_RUNS:
    try:
        significance_results.append(
            compare_models_statistically(BASELINE_RUN, candidate, metric="f1"))
    except (FileNotFoundError, ValueError) as e:
        print_kv("Skipped", f"{candidate}: {e}")

significance_df = pd.DataFrame(significance_results)
if not significance_df.empty:
    significance_path = config.METRICS_DIR / "significance_tests.csv"
    significance_df.to_csv(significance_path, index=False)
    print_kv("Saved", significance_path)
significance_df

In [ ]:
# STAGE 2b - McNemar test + bootstrap CI for the primary 6-variant sweep
# (notebooks/03_training.ipynb Stage 8d, run_name="primary_detection_<model>").
# Wilcoxon (Stage 2) answers "is the difference real across folds"; McNemar
# answers the complementary per-utterance question directly from paired
# predictions, and the bootstrap CI turns a single pooled metric into an
# interval instead of a point estimate. Covers exactly the four ablation
# questions the primary sweep exists to answer:
#   RQ3 fusion vs. either single pathway   fusion          vs acoustic, deep_lora
#   RQ4 LoRA vs. frozen, standalone        deep_lora       vs deep_frozen
#   RQ4 LoRA vs. frozen, inside fusion     fusion          vs fusion_frozen
#   proposed model vs. simple fusion       attention_fusion vs fusion
from src.results import bootstrap_ci, mcnemar_test

PRIMARY_RUN_PREFIX = "primary_detection_"
PRIMARY_PAIRS = [
    ("fusion", "acoustic"), ("fusion", "deep_lora"),
    ("deep_lora", "deep_frozen"), ("fusion", "fusion_frozen"),
    ("attention_fusion", "fusion"),
]

mcnemar_results = []
for better, worse in PRIMARY_PAIRS:
    try:
        mcnemar_results.append(mcnemar_test(
            f"{PRIMARY_RUN_PREFIX}{better}", f"{PRIMARY_RUN_PREFIX}{worse}"))
    except (FileNotFoundError, ValueError) as e:
        print_kv("Skipped", f"{better} vs {worse}: {e}")

mcnemar_df = pd.DataFrame(mcnemar_results)
if not mcnemar_df.empty:
    mcnemar_path = config.METRICS_DIR / "mcnemar_tests.csv"
    mcnemar_df.to_csv(mcnemar_path, index=False)
    print_kv("Saved", mcnemar_path)

bootstrap_results = []
for model in ("acoustic", "deep_frozen", "deep_lora", "fusion_frozen", "fusion", "attention_fusion"):
    try:
        bootstrap_results.append(bootstrap_ci(f"{PRIMARY_RUN_PREFIX}{model}", metric="f1"))
    except (FileNotFoundError, ValueError) as e:
        print_kv("Skipped", f"{model}: {e}")

bootstrap_df = pd.DataFrame(bootstrap_results)
if not bootstrap_df.empty:
    bootstrap_path = config.METRICS_DIR / "bootstrap_ci.csv"
    bootstrap_df.to_csv(bootstrap_path, index=False)
    print_kv("Saved", bootstrap_path)

print_header("McNemar Test (primary sweep pairs)")
display(mcnemar_df)
print_header("Bootstrap 95% CI on F1 (primary sweep, fold-resampled)")
display(bootstrap_df)

In [ ]:
# STAGE 3 - Detection comparison, gated.
#
# This used to read outputs/metrics/phase2_comparison.csv unconditionally.
# That file is written by notebooks/03_training.ipynb from whatever finished,
# with no record of how much finished - in the pre-repair run it contained a
# single held-out control speaker reported as 99.7% accuracy with F1 = 0.
# It is now treated as an INPUT TO VALIDATION, not as a result: the table
# below is built from runs the gate accepted.
detection_final = final_results[final_results["task"] == "detection"]

print_header("Detection comparison — FINAL runs only")
if detection_final.empty:
    print_note("No detection run has passed the gate, so there is no reportable "
              "detection comparison. The legacy phase2_comparison.csv is left on "
              "disk untouched but is NOT read here — see the EXCLUDED table above "
              "for why its rows do not qualify.")
else:
    score_cols = ["accuracy", "precision", "recall", "specificity", "f1", "auroc"]
    display(detection_final.set_index("run_name")[score_cols].style.format(
        "{:.4f}", na_rep="N/A").background_gradient(cmap="RdYlGn", axis=0))
    print_kv("Runs compared", len(detection_final))
    print_kv("All at equal coverage",
             bool(detection_final["coverage"].nunique() == 1))
    if detection_final["coverage"].nunique() > 1:
        print_note("Runs in this table have DIFFERENT fold coverage. They were "
                  "evaluated on different held-out populations and are not "
                  "directly comparable — read them individually, not as a ranking.")

In [ ]:
# STAGE 3b - Ablation gains: the deltas the sweep exists to answer.
#
#   fusion_over_mfcc            fusion           - acoustic         (RQ3)
#   fusion_over_wav2vec         fusion           - deep_lora        (RQ3)
#   lora_over_frozen            deep_lora        - deep_frozen      (RQ4 standalone)
#   lora_over_frozen_in_fusion  fusion           - fusion_frozen    (RQ4 in fusion)
#   proposed_over_fusion        attention_fusion - fusion           (the claim)
#
# Gated: computed from FINAL runs only, and only when both sides of a
# comparison passed. A delta between a complete run and a one-fold run is not
# an ablation result, it is an artifact of unequal evaluation - which is
# exactly what the pre-repair lora_ablation_deltas.csv contained (a +17.8pp
# "accuracy gain" between two single-control-speaker folds, with an F1 delta
# of 0.0 because both F1s were undefined).
from src.results import compute_ablation_gains

print_header("Ablation gains — FINAL runs only")
if detection_final.empty:
    ablation_gains = pd.DataFrame()
    print_note("No FINAL detection runs, so no ablation deltas can be computed.")
else:
    # compute_ablation_gains keys on MODEL name; final_results carries RUN
    # names, so strip the protocol prefix to index it. Runs whose model cannot
    # be recovered are dropped rather than guessed at.
    by_model = detection_final.copy()
    by_model["model"] = (by_model["run_name"]
                         .str.replace(r"^(primary_)?detection_(screen_)?", "", regex=True))
    by_model = by_model.drop_duplicates("model").set_index("model")

    ablation_gains = compute_ablation_gains(by_model)
    if ablation_gains.empty:
        print_note("No comparison had both of its sides in the FINAL table. Each "
                  "delta needs two complete runs on the same protocol.")
    else:
        gains_path = config.RESULTS_DIR / "ablation_gains.csv"
        ablation_gains.to_csv(gains_path, index=False, na_rep="N/A")
        print_kv("Saved", gains_path)
        display(ablation_gains)
        print_note("Absolute gains are percentage points; relative gains are a "
                  "percentage of the baseline. Read these alongside Stage 2b's "
                  "McNemar p-values — a positive delta inside the noise band is "
                  "not evidence of an improvement.")

In [ ]:
# STAGE 4 - Severity comparison, gated.
#
# Severity folds differ from detection in a way that matters here: each holds
# out one speaker PER CLASS, so even a single completed fold covers all four
# severity levels and its metrics are defined. That is why the pre-repair
# severity result (deep_lora, 56.8% accuracy / 0.576 F1 / 0.832 AUROC) is
# genuinely interesting where the detection numbers were not.
#
# It is still only 1 of 20 intended folds, so it lands in PRELIMINARY, not
# FINAL. Shown here, labelled, and not claimable.
severity_final = final_results[final_results["task"] == "severity"]
severity_preliminary = (preliminary_results[preliminary_results["task"] == "severity"]
                        if not preliminary_results.empty else pd.DataFrame())

print_header("Severity comparison")

if not severity_final.empty:
    score_cols = ["accuracy", "precision", "recall", "specificity", "f1", "auroc"]
    print_kv("FINAL severity runs", len(severity_final))
    display(severity_final.set_index("run_name")[score_cols].style.format(
        "{:.4f}", na_rep="N/A").background_gradient(cmap="RdYlGn", axis=0))
else:
    print_kv("FINAL severity runs", "none")

if not severity_preliminary.empty:
    print_header("Severity — PRELIMINARY (not for final claims)")
    print_table(severity_preliminary[["run_name", "completed_folds", "expected_folds",
                                      "accuracy", "f1", "auroc", "reason"]])
    print_note("Each severity fold holds out one speaker per class, so these "
              "metrics ARE defined — unlike a single-speaker detection fold. What "
              "they lack is fold coverage: with 1 of 20 folds there is no estimate "
              "of variance across held-out speaker combinations, so a difference "
              "against the baseline cannot be distinguished from fold luck.")

if severity_final.empty and severity_preliminary.empty:
    print_note("No severity run is registered. notebooks/03_training.ipynb Stages "
              "10-12 train severity only for whichever variant won detection, so "
              "they are no-ops until a detection run completes.")

In [ ]:
# STAGE 5 - ROC and PR curves overlaid across every FINAL detection run.
#
# BUGFIX: this cell previously did
#     DETECTION_RUNS = [n for n in detection_table.index if n != "baseline_svm"]
# where detection_table was indexed by MODEL name ("deep_lora") while
# predictions live under the RUN name ("detection_deep_lora"). load_run_
# predictions therefore raised FileNotFoundError and the notebook died here.
# final_results carries real run names, so the lookup now resolves.
from src.results import plot_roc_pr_comparison

DETECTION_RUNS = [r for r in detection_final["run_name"] if "baseline_svm" not in r]

print_header("ROC / PR Comparison")
if not DETECTION_RUNS:
    curve_paths = None
    print_note("No FINAL detection run to plot. ROC and PR curves need a pooled "
              "held-out set containing both classes — the gate has already "
              "excluded anything that does not.")
else:
    curve_paths = plot_roc_pr_comparison(DETECTION_RUNS, task="detection", show=True)
    print_kv("Runs plotted", ", ".join(DETECTION_RUNS))
    print_kv("ROC figure", curve_paths["roc"])
    print_kv("PR figure", curve_paths["pr"])

In [ ]:
# STAGE 6 - Publication figures: model comparison, confusion matrices,
# coverage, and the paper-vs-reproduction-vs-proposed baseline chart.
#
# BUGFIX: BEST_RUN was previously `detection_table["f1"].idxmax()`. With F1
# written as 0.0 for every single-class fold, that was an argmax over a
# column of zeros - it returned whichever row sorted first, and called it the
# best model. It now selects among FINAL runs only, and skips NaN.
import matplotlib.pyplot as plt

from src.results import plot_baseline_comparison, plot_fold_coverage

# --- Figure: fold coverage. Deliberately unconditional. This is the figure
# that makes incomplete experiments impossible to overlook, so it renders
# whatever the state of the project - including "almost nothing ran".
coverage_figure = plot_fold_coverage(show=True)
print_kv("Coverage figure", coverage_figure or "no registered runs yet")

# --- Best FINAL run and its pooled confusion matrix
scored = detection_final[detection_final["f1"].notna()]
BEST_RUN = scored.iloc[0]["run_name"] if len(scored) else None

if BEST_RUN is None:
    print_note("No FINAL detection run with a defined F1 — no best model to report.")
else:
    print_kv("Best FINAL detection run (by pooled F1)",
             f"{BEST_RUN} (F1={scored.iloc[0]['f1']:.4f})")
    cm_path = config.CONFUSION_MATRIX_DIR / BEST_RUN / "ALL_FOLDS_pooled.png"
    if cm_path.exists():
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.imshow(plt.imread(cm_path))
        ax.axis("off")
        ax.set_title(f"{BEST_RUN} — pooled confusion matrix")
        plt.show()

# --- Figure: paper vs reproduction vs proposed.
# The paper's 93.95% is a CITATION, not a measurement from this pipeline.
# plot_baseline_comparison colours the three provenances differently for
# exactly that reason - mixing them is how a reproduction gap silently turns
# into a performance claim.
PAPER_DETECTION_ACCURACY = 0.9395      # Javanmardi et al., ICASSP 2023, layer 1

reproduction_accuracy = None
sweep_path = config.METRICS_DIR / "baseline_svm_detection_layer_sweep.csv"
if sweep_path.exists():
    reproduction_accuracy = float(pd.read_csv(sweep_path).iloc[0]["accuracy"])

proposed = {row["run_name"]: row["accuracy"]
            for _, row in detection_final.iterrows()
            if "baseline_svm" not in row["run_name"]}

baseline_figure = plot_baseline_comparison(
    reproduction_accuracy=reproduction_accuracy, proposed=proposed,
    paper_accuracy=PAPER_DETECTION_ACCURACY, task="detection", show=True)

print_header("Publication Figures")
print_kv("Baseline comparison", baseline_figure or "needs at least a reproduction number")
if reproduction_accuracy is not None:
    gap = PAPER_DETECTION_ACCURACY - reproduction_accuracy
    print_kv("Reproduction vs paper", f"{reproduction_accuracy:.4f} vs "
             f"{PAPER_DETECTION_ACCURACY:.4f}  (gap {gap:+.4f})")
    print_note("This gap is REPORTED, not tuned away. See notebooks/04's baseline "
              "reproduction diagnostic for the component-by-component breakdown; "
              "the leading candidate is unmasked mean-pooling over a window that "
              "is ~86% zero-padding on a median utterance.")

In [ ]:
# STAGE 7 - Gated export + the research summary.
#
# Writes outputs/results/:
#   final_results.csv         only runs that passed the gate
#   preliminary_results.csv   real but incomplete, labelled as such
#   excluded_results.csv      every exclusion, with its reason
#   experiment_manifest.csv   reproducibility: model, task, protocol, seed,
#                             epochs, LR, batch size, checkpoint path, plus
#                             the dataset/VAD/MFCC/LoRA configuration
#   *.png                     coverage and baseline-comparison figures
#
# Undefined metrics export as the string "N/A", never 0 — a reader of the CSV
# must not be able to mistake "not measurable" for "measured as zero".
#
# export_results_for_paper() (the older copy-and-format step into
# outputs/paper_exports/) still runs afterwards for the LaTeX snippets, but
# outputs/results/ is the gated source of truth.
from src.results import export_gated_results, export_results_for_paper, print_experiment_summary

print_header("Gated export")
written = export_gated_results(MIN_COVERAGE)

if DETECTION_RUNS:
    export_dir = export_results_for_paper(run_names=DETECTION_RUNS)
    print_kv("Paper export directory", export_dir)
else:
    print_note("Skipped outputs/paper_exports/ — nothing has passed the gate, so "
              "there is nothing to hand off for the paper yet.")

# ---------------------------------------------------------------------------
# The summary. Every value comes from artifacts on disk. A model that was
# never trained is reported as "NOT EXECUTED" rather than omitted — an absent
# row is precisely how the attention-fusion variants disappeared from the
# pre-repair report without anyone noticing.
# ---------------------------------------------------------------------------
print_experiment_summary(MIN_COVERAGE)

print_header("Reporting gate — what this run may claim")
if final_results.empty:
    print_note("NOTHING. No experiment has passed the eligibility gate. This is a "
              "true statement about the project's current state, not a failure of "
              "this notebook: the training stages have not yet produced a complete, "
              "class-covering evaluation of any variant.")
    print_note("Next: notebooks/03_training.ipynb in DEVELOPMENT mode, which runs "
              "every variant on one shared class-balanced protocol.")
else:
    print_kv("Claimable results", len(final_results))
    print_kv("Files", ", ".join(p.name for p in written.values()))